# M2 학습예산 진단 — 개인 구매이력 기반 N/V 표현

이번 단계는 **M1과 M2 두 arm만 300 epoch까지 학습**하고, 25 epoch마다 개발분할을 평가합니다. 질문은 하나입니다.

> 기존 100 epoch에서 M2 학습이 너무 일찍 종료되어 M1 대비 격차가 남았는가?

고정 조건은 ID 64차원, N/V 축별 4차원, L2 `1e-3`, `rho=0.05`, 2층, K=1 균일 음성입니다. M2는 사용자의 `q_N`·`q_V`가 개인 구매이력–후보상품 적합 블록을 조절하며, 같은 BPR 손실과 optimizer 안에서 학습됩니다. 외부 재정렬은 없습니다.

**범위 주의:** 이 M2는 CLV의 N/V 구성요소를 모두 사용하지만 `q_C=percentile(N×V)`는 사용하지 않습니다. 따라서 전체 historical CLV 수준 모형이 아니라 **CLV 구성요소 기반 M2**로 판독합니다.

매 평가 시점에 다음을 저장합니다.

- Recall/NDCG 및 가격·구매금액 가중 적중 지표
- 동일 epoch의 `M2−M1`
- 평가사용자 Top-10에서 N/V 블록이 점수에 차지하는 비중
- ID 및 N/V 임베딩 gradient
- epoch별 체크포인트와 자동 재개 상태

이번 노트북에서는 epoch를 선택하지 않습니다. 단일 seed 42 학습곡선으로 방향만 확인하고, 후속 조건이나 다중 seed는 결과를 본 뒤 별도 사전등록합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '211ce6f654217e01495bf451fe5505567e8607c6'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m2_capacity_search as search

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = search.configure_capacity_search(conditions=('baseline',))
summary = search.preflight_summary(cfg)
assert summary['protocol_epoch'] in summary['evaluated_at_epochs']
assert summary['m1_retrained_per_shared_setting'] is True
assert list(summary['conditions']) == ['baseline']
assert search.arm_specifications(cfg)[0]['model_id'] == search.M1_MODEL_ID
assert search.arm_specifications(cfg)[1]['model_id'] == search.M2_MODEL_ID
assert len(search.arm_specifications(cfg)) == 2
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
curve = search.run_capacity_search(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy(); view.attrs = {}; display(view)

print('1) 조건별 학습곡선 (25 epoch 간격, clv_score_share = CLV 블록이 점수에서 차지하는 비중)')
show(curve)
print('2) 같은 조건·같은 시드에서 M2 - M1')
show(search.gap_frame(curve))
print('3) 판독')
print(json.dumps(curve.attrs['reading'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(curve.attrs['result_paths'], ensure_ascii=False, indent=2))


In [ ]:
# 교수님 보고용 그림: 개발 성능이 100 epoch 이후에도 오르는가
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, part in curve.groupby('condition'):
    for model_id, line in part.groupby('model_id'):
        axes[0].plot(line.epoch, line['recall@10'], marker='o', markersize=3,
                     label=f'{name} / {model_id.split("_")[0]}')
for name, part in search.gap_frame(curve).groupby('condition'):
    axes[1].plot(part.epoch, part['recall@10'], marker='o', markersize=3, label=name)
for ax, title in zip(axes, ['개발 Recall@10', 'M2 - M1 (Recall@10)']):
    ax.axvline(100, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('epoch'); ax.set_title(title); ax.legend(fontsize=7)
axes[1].axhline(0, color='black', linewidth=0.8)
plt.tight_layout(); plt.show()
